# Проверка портфеля — ТЧК MISIS

Ноутбук вызывает **тот же движок, что UI и API**: формулы организаторов, диагностику ограничений и `hybrid_maximin_v1`.
Сначала показывает сохранённый результат команды, затем позволяет менять входы и запускать новый расчёт кнопкой.
Сервер приложения, Docker, Ollama и ключи API не требуются. После установки зависимостей работает без интернета.

**Запуск из корня репозитория или автономного комплекта** (Python 3.12):
```bash
python3.12 -m venv .venv
.venv/bin/python -m pip install -r notebooks/requirements.txt
.venv/bin/python -m jupyterlab notebooks/portfolio_review.ipynb
```
В Windows замените `.venv/bin/python` на `.venv\Scripts\python.exe`.
В Jupyter выберите **Run → Run All Cells**, затем используйте поля и кнопки ниже. Сам Run All не запускает перебор.
Исходные файлы и `results/` не перезаписываются; изменения и история расчётов живут в текущем ядре Jupyter.

In [ ]:
from copy import deepcopy
from pathlib import Path
import importlib
import json
import sys

import pandas as pd
import ipywidgets as W
from IPython.display import display, Markdown, clear_output

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "config/decision.json").is_file()), None)
if ROOT is None:
    raise RuntimeError("Откройте ноутбук из каталога репозитория или автономного комплекта.")
standalone = (ROOT / "src/engine/canonical.py").is_file()
sys.path.insert(0, str(ROOT / "src" if standalone else ROOT))
PACKAGE = "engine" if standalone else "backend.app.core.services.portfolio_engine"
canonical = importlib.import_module(f"{PACKAGE}.canonical")
hybrid = importlib.import_module(f"{PACKAGE}.hybrid")
diagnose = importlib.import_module(f"{PACKAGE}.constraints").diagnose
verify_export = importlib.import_module(f"{PACKAGE}.export_integrity").verify_export
verify_export(ROOT / "results", ROOT / "config/decision.json")

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

config = read_json(ROOT / "config/decision.json")
saved = read_json(ROOT / "results/team_decision_config.json")
saved_metrics = read_json(ROOT / "results/portfolio_metrics.json")
saved_analysis = read_json(ROOT / "results/hybrid_analysis.json")
lots, modes, official_limits = canonical.load_case()
official_inputs = {"lots": lots.to_dict("records"), "modes": modes.to_dict("records")}
for row in official_inputs["lots"]:
    row["capability_groups"] = row["capability_groups"].split(";")
initial_inputs = deepcopy(config["algorithm_parameters"].get("inputs") or official_inputs)
baseline_inputs = deepcopy(saved["algorithm_parameters"].get("inputs") or official_inputs)
baseline = dict(name="Сохранённое решение команды", selection=saved["recommended"]["selection"],
                inputs=baseline_inputs, metrics=saved_metrics)
history = [baseline]
latest = None
print("Хеши исходных файлов, движка и четырёх результатов совпали. Перебор ещё не запускался.")
print("Версия данных:", canonical.source_version())

## 1. Сохранённый результат

Это чтение четырёх файлов сдачи, **не новый расчёт**. Ниже видны состав и режимы, показатели,
шесть оценок выбора и проверка одного состава в BASE/STRESS. У STRESS меняется только лимит запуска.

In [ ]:
def show_checks(checks):
    display(pd.DataFrame([dict(scenario=scenario, **row)
                          for scenario, rows in checks.items() for row in rows]))

display(pd.DataFrame(baseline["selection"], columns=["lot_id", "mode_id"]))
display(pd.DataFrame([saved_metrics]))
display(pd.DataFrame(saved_analysis["winner"]["components"]))
show_checks(saved_analysis["checks"])

## 2. Входные данные

Раскройте нужный лот или режим. Название поля совпадает с контрактом данных; при наведении появляется расшифровка.
Все восемь лотов и режимы A/B/C доступны для редактирования. Их идентификаторы и обязательные лимиты кейса фиксированы.
Изменённые цены, индексы, признаки и коэффициенты — **ваш эксперимент**, а не новые данные организаторов.
Коэффициенты режима применяются ко всем лотам, которым он назначен.

In [ ]:
FIELD_HELP = {
    "service": "Название сервиса", "territorial_archetype": "Территориальный архетип",
    "capability_groups": "Группы возможностей через ;: EO, PNT/InSAR, SATCOM, SSA",
    "federal": "Федеральный лот: не добавляет территориальный архетип",
    "c0_mrub": "Разовые затраты запуска, млн ₽",
    "opex_mrub_per_year": "Ежегодные расходы, млн ₽/год",
    "anchor_cash_mrub_per_year": "Якорные поступления, млн ₽/год",
    "commercial_cash_mrub_per_year": "Коммерческие поступления, млн ₽/год",
    "vpub_mrub_per_year": "Общественная ценность, млн ₽/год; не выручка",
    "t_rep": "Показатель кейса 0–1; используется только как отдельное ограничение",
    "readiness_1_5": "Индекс готовности, от 1 до 5",
    "resilience_1_5": "Индекс устойчивости, от 1 до 5; не вероятность",
    "scale_1_5": "Индекс тиражируемости, от 1 до 5",
    "k_c0": "Множитель затрат запуска", "k_opex": "Множитель ежегодных расходов",
    "k_vpub": "Множитель общественной ценности", "k_anchor": "Множитель якорных поступлений",
    "k_commercial": "Множитель коммерческих поступлений",
    "public_core": "Режим засчитывается в общественное ядро",
}
editors = {}

def make_editor(collection, row):
    key = "lot_id" if collection == "lots" else "mode_id"
    controls, boxes = {}, []
    for field, value in row.items():
        if field == key:
            continue
        if isinstance(value, bool):
            control = W.Checkbox(value=value, indent=False)
        elif isinstance(value, (int, float)):
            control = W.FloatText(value=value)
        else:
            control = W.Text(value=";".join(value) if isinstance(value, list) else value)
        control.layout.width = "100%"
        controls[field] = control
        label = W.Label(value=field, tooltip=FIELD_HELP[field])
        boxes.append(W.VBox([label, control], layout=W.Layout(width="310px")))
    editors[(collection, row[key])] = controls
    return W.Box(boxes, layout=W.Layout(display="flex", flex_flow="row wrap", gap="8px 16px"))

panels = []
for collection, key in (("lots", "lot_id"), ("modes", "mode_id")):
    rows = initial_inputs[collection]
    panel = W.Accordion(children=[make_editor(collection, row) for row in rows], selected_index=None)
    for i, row in enumerate(rows):
        panel.set_title(i, row[key])
    panels.append(panel)
tabs = W.Tab(children=panels)
tabs.set_title(0, "Лоты")
tabs.set_title(1, "Режимы A/B/C")

def collect_inputs():
    inputs = deepcopy(initial_inputs)
    for collection, key in (("lots", "lot_id"), ("modes", "mode_id")):
        for row in inputs[collection]:
            for field, control in editors[(collection, row[key])].items():
                row[field] = ([v.strip() for v in control.value.split(";") if v.strip()]
                              if field == "capability_groups" else control.value)
    canonical.validate_inputs(inputs)
    return inputs

def set_inputs(inputs):
    for collection, key in (("lots", "lot_id"), ("modes", "mode_id")):
        for row in inputs[collection]:
            for field, control in editors[(collection, row[key])].items():
                value = row[field]
                control.value = ";".join(value) if isinstance(value, list) else value

reset = W.Button(description="Вернуть входы конфигурации", layout=W.Layout(width="270px"))
reset.on_click(lambda _: set_inputs(initial_inputs))
display(tabs, reset)

## 3. Новый расчёт

**Автоподбор** рассматривает все лоты и режимы A/B/C на текущих входах. Состав выбирает алгоритм.
Сначала отсекает нарушения, затем выбирает наибольшую минимальную оценку Q, при равенстве — больший остаток S.
Нормировка 0–1 сравнивает положение каждого показателя между худшим и лучшим допустимым значением;
для затрат запуска направление обратное. Это правило защищает слабейшую относительную оценку, но не гарантирует максимум каждого показателя.

**Ручная проверка** считает заданные лоты и режимы. Выберите ровно четыре: иначе увидите нарушение требования о составе.
Ручные переключатели не ограничивают автоподбор. В обоих случаях проверяется один полученный состав в BASE и STRESS.

Автоматический Δ означает приоритет Q → S. Ручной Δ ограничивает потерю годового остатка относительно его максимума.
Дополнительные пороги могут только ужесточить условия кейса. Дополнительная чувствительность включается отдельно и не меняет выбор.

In [ ]:
params = config["algorithm_parameters"]
operation = W.ToggleButtons(options=[("Автоподбор", "auto"), ("Ручная проверка", "manual")])
stress = W.Checkbox(value=params.get("require_stress", True), description="Искать только проходящие STRESS", indent=False, layout=W.Layout(width="350px"))

def optional_number(title, value):
    enabled = W.Checkbox(value=value is not None, description=title, indent=False, layout=W.Layout(width="350px"))
    number = W.FloatText(value=value if value is not None else 0, disabled=not enabled.value)
    enabled.observe(lambda change: setattr(number, "disabled", not change["new"]), names="value")
    return enabled, number, W.HBox([enabled, number])

delta_on, delta, delta_box = optional_number("Задать Δ, млн ₽/год", params.get("cash_loss_limit_mrub"))
budget_on, budget, budget_box = optional_number("Свой предел C0, млн ₽", params.get("budget_cap_mrub"))
vpub_on, vpub, vpub_box = optional_number("Свой минимум VPUB, млн ₽/год", params.get("vpub_floor_mrub_per_year"))
sensitivity = W.Checkbox(value=False, description="Дополнительные сценарии чувствительности", indent=False, layout=W.Layout(width="400px"))
auto_panel = W.VBox([stress, delta_box, budget_box, vpub_box, sensitivity])
baseline_modes = dict(baseline["selection"])
manual_modes = {row["lot_id"]: W.Dropdown(options=[("Не включать", ""), ("A", "A"), ("B", "B"), ("C", "C")],
                value=baseline_modes.get(row["lot_id"], ""), description=row["lot_id"], layout=W.Layout(width="230px"))
                for row in initial_inputs["lots"]}
manual_panel = W.Box(list(manual_modes.values()), layout=W.Layout(display="none", flex_flow="row wrap"))

def switch_operation(change):
    auto_panel.layout.display = "flex" if change["new"] == "auto" else "none"
    manual_panel.layout.display = "flex" if change["new"] == "manual" else "none"

operation.observe(switch_operation, names="value")
run = W.Button(description="Рассчитать", button_style="primary", icon="play")
status = W.HTML(value="Расчёт ещё не запускался.")
output = W.Output()

def changed_fields(before, after):
    rows = []
    for collection, key in (("lots", "lot_id"), ("modes", "mode_id")):
        old = {row[key]: row for row in before[collection]}
        for row in after[collection]:
            for field, value in row.items():
                if value != old[row[key]][field]:
                    rows.append(dict(id=row[key], field=field, before=old[row[key]][field], after=value))
    return rows

def comparison_row(item):
    metrics = item["metrics"]
    return dict(variant=item["name"], selection=" · ".join(f"{lot}:{mode}" for lot, mode in item["selection"]),
                **metrics, S=metrics["cash_mrub_per_year"] - metrics["opex_mrub_per_year"],
                BASE=all(canonical.canonical_checks(metrics, "BASE").values()),
                STRESS=all(canonical.canonical_checks(metrics, "STRESS").values()))

def calculate(_=None):
    global latest
    run.disabled = True
    latest = None
    status.value = "Считаем…"
    with output:
        clear_output(wait=True)
        try:
            inputs = collect_inputs()
            engine_inputs = None if inputs == official_inputs else inputs
            analysis = None
            if operation.value == "auto":
                parameters = hybrid.Parameters(inputs=engine_inputs, require_stress=stress.value,
                    cash_loss_limit_mrub=delta.value if delta_on.value else None,
                    budget_cap_mrub=budget.value if budget_on.value else None,
                    vpub_floor_mrub_per_year=vpub.value if vpub_on.value else None)
                _, _, analysis = hybrid.analyze(parameters, include_sensitivity=sensitivity.value)
                if analysis["winner"] is None:
                    status.value = "Допустимого портфеля нет. Проверьте входы и заданные пороги."
                    print("При текущих условиях ни один вариант не проходит. Предыдущий результат не используется.")
                    return
                selection = [(row["lot_id"], row["mode_id"]) for row in analysis["winner"]["selection"]]
            else:
                selection = [(lot, widget.value) for lot, widget in manual_modes.items() if widget.value]
                if len(selection) > 4:
                    raise ValueError("Для ручной проверки выберите не более четырёх лотов; итоговое требование кейса — ровно четыре.")
            detail, metrics = canonical.evaluate(selection, engine_inputs)
            checks = {scenario: [row.as_dict() for row in diagnose(metrics, scenario)] for scenario in ("BASE", "STRESS")}
            latest = dict(name=f"Расчёт {len(history)} · {operation.label}", selection=selection,
                          inputs=deepcopy(inputs), metrics=metrics, analysis=analysis, checks=checks)
            history.append(deepcopy(latest))
            display(Markdown("### Состав и расчёт по лотам"))
            display(detail)
            display(Markdown("### Сравнение с сохранёнными результатами"))
            display(pd.DataFrame([comparison_row(item) for item in history]).set_index("variant").T)
            changes = changed_fields(baseline_inputs, inputs)
            display(Markdown("### Изменения входов относительно решения команды"))
            if changes:
                display(pd.DataFrame(changes))
            else:
                print("Входные данные совпадают с сохранённым решением команды.")
            display(Markdown("### Все обязательные проверки"))
            show_checks(checks)
            if analysis:
                display(Markdown("### Почему выбран этот состав"))
                display(pd.DataFrame(analysis["winner"]["components"]))
                display(pd.DataFrame([{key: analysis[key] for key in
                    ("reference_count", "feasible_count", "max_q_count", "q_max", "s_max_mrub", "effective_delta_mrub")}]))
                print("0 — худшее, 1 — лучшее в допустимой области текущих входов. Q — минимальная из шести оценок.")
                print("Q разных входных наборов нельзя напрямую сравнивать: границы шкал могут измениться.")
                if analysis["sensitivity"]:
                    display(Markdown("### Отдельные сценарии чувствительности"))
                    display(pd.DataFrame([dict(title=row["title"], feasible_count=row["feasible_count"],
                        winner_changed=row["outcome"]["winner_changed"],
                        original_still_feasible=row["outcome"]["original_still_feasible"])
                        for row in analysis["sensitivity"]]))
            status.value = "Готово. Показан снимок расчёта; после изменения полей нажмите «Рассчитать» снова."
        except (ValueError, KeyError, TypeError) as error:
            status.value = "Расчёт не выполнен."
            print(str(error))
        finally:
            run.disabled = False

def mark_stale(_):
    status.value = "Поля изменены. Нажмите «Рассчитать»: ранее показанные результаты относятся к прежним входам."

for controls in editors.values():
    for control in controls.values():
        control.observe(mark_stale, names="value")
for control in [operation, stress, delta_on, delta, budget_on, budget, vpub_on, vpub, sensitivity, *manual_modes.values()]:
    control.observe(mark_stale, names="value")
run.on_click(calculate)
display(operation, auto_panel, manual_panel, run, status, output)

## Как проверить изменения

1. Нажмите **«Рассчитать»** с исходными настройками: новый автоподбор можно сравнить с сохранённым решением выше.
2. Измените стоимость или коэффициент режима в разделе 2 и нажмите кнопку снова. Появится новый столбец показателей и перечень изменённых входов.
3. Переключитесь на **«Ручная проверка»**, измените состав или режимы. Таблица покажет факты, пороги, запасы и PASS/FAIL в обоих сценариях.
4. Чтобы изучить другие риски, включите дополнительные сценарии в автоподборе. Это отдельные допущения, а не изменение официального STRESS.

История хранит собственные входы каждого расчёта. После перезапуска ядра она очищается.
Управленческое обоснование из записки относится к решению команды: при изменении входов и состава его применимость нужно пересмотреть.

**Обозначения:** C0 — вложения на запуск; OPEX — годовые расходы; CASH — годовые поступления;
S = CASH − OPEX — остаток до возврата C0, налогов и стоимости капитала, не чистая прибыль.
VPUB — общественная ценность, она не складывается с CASH. KCASH = CASH / OPEX — покрытие расходов.
Готовность, устойчивость и тиражируемость — индексы `readiness_1_5`, `resilience_1_5`, `scale_1_5`.
Смысл `t_rep` организаторами не раскрыт: используется только его официальный порог.

Код ячеек — подключение движка и интерфейс проверки. Расчётные функции: `canonical.evaluate`, `constraints.diagnose`, `hybrid.analyze`.
Полный сохранённый отчёт — `results/hybrid_analysis.json` в корне проекта/комплекта.
Интерфейс построен на [JupyterLab](https://jupyterlab.readthedocs.io/en/stable/) и
[ipywidgets](https://ipywidgets.readthedocs.io/en/stable/examples/Widget%20List.html) (BSD-3-Clause).